In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import lightgbm as lgb
from prophet import Prophet
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'axes.grid': True, 'grid.alpha': 0.3, 'axes.facecolor': '#f5f5f5'})

In [ ]:
df = pd.read_csv('final_ml_ready.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['product', 'date']).reset_index(drop=True)

FEATURE_COLS = [
    'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12',
    'roll_mean_3', 'roll_mean_6', 'roll_std_3',
    'trend_slope_12', 'quarter',
    'is_lebaran_window', 'is_year_end', 'year_index', 'fc_error_1', 'fc_acc_1', 'usd_change_rate'
]
TARGET = 'actual'

horizon_starts = pd.date_range('2025-03-01', '2026-01-01', freq='MS')
products = df['product'].unique()

print(f'Products: {list(products)}')
print(f'Horizons: {len(horizon_starts)} ({horizon_starts[0].date()} to {horizon_starts[-1].date()})')

In [ ]:
def mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mp = mape(y_true, y_pred)
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'MAPE': mp}

def build_features(actual_series, forecast_series, known_row):
    n = len(actual_series)
    f = {}
    def _safe(v):
        return v if np.isfinite(v) else 0.0

    f['lag_1'] = _safe(actual_series[-1]) if n >= 1 else 0.0
    f['lag_2'] = _safe(actual_series[-2]) if n >= 2 else 0.0
    f['lag_3'] = _safe(actual_series[-3]) if n >= 3 else 0.0
    f['lag_6'] = _safe(actual_series[-6]) if n >= 6 else 0.0
    f['lag_12'] = _safe(actual_series[-12]) if n >= 12 else 0.0
    f['roll_mean_3'] = _safe(np.mean(actual_series[-3:])) if n >= 3 else 0.0
    f['roll_mean_6'] = _safe(np.mean(actual_series[-6:])) if n >= 6 else 0.0
    f['roll_std_3'] = _safe(np.std(actual_series[-3:])) if n >= 3 else 0.0
    f['trend_slope_12'] = _safe(np.polyfit(np.arange(12), actual_series[-12:], 1)[0]) if n >= 12 else 0.0
    if n >= 1:
        a, fc = actual_series[-1], forecast_series[-1]
        fc_err = (a - _safe(fc)) / a if a != 0 else 0
    else:
        fc_err = 0.0
    f['fc_error_1'] = fc_err
    f['fc_acc_1'] = 1 - abs(fc_err)
    for col in ['quarter', 'is_lebaran_window', 'is_year_end', 'year_index', 'usd_change_rate']:
        f[col] = _safe(known_row[col])
    return pd.Series(f)

In [ ]:
def run_wfcv_sklearn(model_fn, prod_data, horizon_starts):
    prod_data = prod_data.sort_values('date').reset_index(drop=True)
    all_horizon_metrics = []

    for h_start in horizon_starts:
        cutoff = h_start - pd.DateOffset(months=1)
        train_df = prod_data[prod_data['date'] <= cutoff].copy()
        pred_dates = pd.date_range(h_start, periods=3, freq='MS')

        train_df = train_df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
        if len(train_df) < 10:
            all_horizon_metrics.append({'MAE': np.nan, 'MSE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan,
                                        'preds': [], 'actuals': [], 'dates': []})
            continue

        model = model_fn()
        model.fit(train_df[FEATURE_COLS], train_df[TARGET])

        actual_series = list(train_df[TARGET].values)
        forecast_series = list(train_df['forecast'].values)
        preds = []
        actuals = []

        for pdate in pred_dates:
            row = prod_data[prod_data['date'] == pdate]
            if len(row) == 0:
                preds.append(np.nan)
                continue
            row = row.iloc[0]
            feat = build_features(actual_series, forecast_series, row)
            X = feat[FEATURE_COLS].values.reshape(1, -1)
            pred = model.predict(X)[0]
            preds.append(pred)
            actuals.append(row[TARGET])
            actual_series.append(pred)
            forecast_series.append(row['forecast'])

        preds = np.array(preds)
        actuals = np.array(actuals)
        mask = ~np.isnan(preds)
        if mask.sum() < 2:
            all_horizon_metrics.append({'MAE': np.nan, 'MSE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan,
                                        'preds': [], 'actuals': [], 'dates': []})
        else:
            h_metrics = compute_metrics(actuals[mask], preds[mask])
            h_metrics['preds'] = list(preds[mask])
            h_metrics['actuals'] = list(actuals[mask])
            h_metrics['dates'] = [d for d, m in zip(pred_dates, mask) if m]
            all_horizon_metrics.append(h_metrics)

    avg = {k: np.nanmean([m[k] for m in all_horizon_metrics]) for k in ['MAE', 'MSE', 'RMSE', 'MAPE']}
    return {'per_horizon': all_horizon_metrics, 'avg': avg}

In [ ]:
def run_wfcv_prophet(prod_data, horizon_starts):
    prod_data = prod_data.sort_values('date').reset_index(drop=True)
    all_horizon_metrics = []

    for h_start in horizon_starts:
        cutoff = h_start - pd.DateOffset(months=1)
        train_df = prod_data[prod_data['date'] <= cutoff].copy()
        pred_dates = pd.date_range(h_start, periods=3, freq='MS')

        train_df = train_df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
        if len(train_df) < 10:
            all_horizon_metrics.append({'MAE': np.nan, 'MSE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan,
                                        'preds': [], 'actuals': [], 'dates': []})
            continue

        prophet_df = train_df[['date', TARGET] + FEATURE_COLS].rename(columns={'date': 'ds', TARGET: 'y'})
        m = Prophet()
        for col in FEATURE_COLS:
            m.add_regressor(col)
        m.fit(prophet_df)

        actual_series = list(train_df[TARGET].values)
        forecast_series = list(train_df['forecast'].values)
        preds = []
        actuals = []

        for pdate in pred_dates:
            row = prod_data[prod_data['date'] == pdate]
            if len(row) == 0:
                preds.append(np.nan)
                continue
            row = row.iloc[0]
            feat = build_features(actual_series, forecast_series, row)
            future = pd.DataFrame({'ds': [pdate]})
            for col in FEATURE_COLS:
                future[col] = feat[col]
            forecast = m.predict(future)
            pred = forecast['yhat'].values[0]
            preds.append(pred)
            actuals.append(row[TARGET])
            actual_series.append(pred)
            forecast_series.append(row['forecast'])

        preds = np.array(preds)
        actuals = np.array(actuals)
        mask = ~np.isnan(preds)
        if mask.sum() < 2:
            all_horizon_metrics.append({'MAE': np.nan, 'MSE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan,
                                        'preds': [], 'actuals': [], 'dates': []})
        else:
            h_metrics = compute_metrics(actuals[mask], preds[mask])
            h_metrics['preds'] = list(preds[mask])
            h_metrics['actuals'] = list(actuals[mask])
            h_metrics['dates'] = [d for d, m in zip(pred_dates, mask) if m]
            all_horizon_metrics.append(h_metrics)

    avg = {k: np.nanmean([m[k] for m in all_horizon_metrics]) for k in ['MAE', 'MSE', 'RMSE', 'MAPE']}
    return {'per_horizon': all_horizon_metrics, 'avg': avg}

In [ ]:
models = {
    'RandomForest': lambda: RandomForestRegressor(
        n_estimators=200, max_depth=10, min_samples_leaf=3, random_state=42, n_jobs=-1
    ),
    'XGBoost': lambda: xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42
    ),
    'Lasso': lambda: make_pipeline(
        StandardScaler(), Lasso(alpha=0.01, random_state=42, max_iter=10000)
    ),
    'LightGBM': lambda: lgb.LGBMRegressor(
        n_estimators=200, max_depth=8, learning_rate=0.1, random_state=42, verbose=-1
    ),
}

all_results = {}

for prod in products:
    print(f'\n========== {prod} ==========')
    prod_data = df[df['product'] == prod].copy()
    all_results[prod] = {}

    for name, fn in models.items():
        res = run_wfcv_sklearn(fn, prod_data, horizon_starts)
        all_results[prod][name] = res
        print(f'  {name:12s}  MAPE={res["avg"]["MAPE"]:7.2f}  MAE={res["avg"]["MAE"]:9.0f}  RMSE={res["avg"]["RMSE"]:9.0f}')

    res = run_wfcv_prophet(prod_data, horizon_starts)
    all_results[prod]['Prophet'] = res
    print(f'  {"Prophet":12s}  MAPE={res["avg"]["MAPE"]:7.2f}  MAE={res["avg"]["MAE"]:9.0f}  RMSE={res["avg"]["RMSE"]:9.0f}')

In [ ]:
model_names = ['RandomForest', 'XGBoost', 'Lasso', 'LightGBM', 'Prophet']
metrics_keys = ['MAE', 'MSE', 'RMSE', 'MAPE']

for prod in products:
    print(f'\n=== {prod} - Avg WFCV Metrics ({len(horizon_starts)} horizons x 3 months) ===')
    rows = []
    for m in model_names:
        r = all_results[prod][m]['avg']
        rows.append({'Model': m, **{k: f'{r[k]:.2f}' for k in metrics_keys}})
    print(pd.DataFrame(rows).set_index('Model').to_string())

In [ ]:
print('=== OVERALL (averaged across all 4 products) ===')
overall = {m: {k: [] for k in metrics_keys} for m in model_names}
for prod in products:
    for m in model_names:
        for k in metrics_keys:
            overall[m][k].append(all_results[prod][m]['avg'][k])

rows = []
for m in model_names:
    row = {'Model': m}
    for k in metrics_keys:
        row[k] = f'{np.mean(overall[m][k]):.2f}'
    rows.append(row)
print(pd.DataFrame(rows).set_index('Model').to_string())

In [ ]:
n_horizons = len(horizon_starts)
prod = products[0]
print(f'=== {prod} - Per-Horizon MAPE ===')
data = {}
for m in model_names:
    data[m] = [h['MAPE'] for h in all_results[prod][m]['per_horizon']]
data['Horizon'] = [f'H{i+1}' for i in range(n_horizons)]
df_h = pd.DataFrame(data).set_index('Horizon')
print(df_h.round(2).to_string())

In [ ]:
n_horizons = len(horizon_starts)
plot_months = pd.date_range('2025-03-01', '2026-03-01', freq='MS')

for prod in products:
    plt.figure(figsize=(16, 6))

    actuals = df[df['product'] == prod].set_index('date').loc[plot_months, TARGET]
    plt.plot(actuals.index, actuals.values, 'k-', linewidth=2.5, label='Actual')

    best_model = min(model_names, key=lambda m: all_results[prod][m]['avg']['MAPE'])
    best_mape = all_results[prod][best_model]['avg']['MAPE']

    for i in range(n_horizons):
        h = all_results[prod][best_model]['per_horizon'][i]
        if len(h['preds']) >= 2:
            plt.plot(h['dates'], h['preds'],
                     color='red', alpha=0.15, linewidth=1, marker='.', markersize=5)

    pred_series = []
    for i, month in enumerate(plot_months):
        if i < n_horizons:
            h = all_results[prod][best_model]['per_horizon'][i]
            if len(h['preds']) > 0:
                pred_series.append(h['preds'][0])
            else:
                pred_series.append(np.nan)
        else:
            h = all_results[prod][best_model]['per_horizon'][-1]
            step = i - (n_horizons - 1)
            if len(h['preds']) > step:
                pred_series.append(h['preds'][step])
            else:
                pred_series.append(np.nan)

    plt.plot(plot_months, pred_series, 'r-', marker='o', linewidth=1.8,
             label=f"{best_model} (MAPE={best_mape:.1f}%)")

    plt.title(f'{prod} - Best Model: {best_model} vs Actual (WFCV)', fontsize=14, fontweight='bold')
    plt.xlabel('Month', fontsize=12)
    plt.ylabel('Actual Sales', fontsize=12)
    plt.xticks(rotation=45)
    plt.legend(fontsize=11)
    plt.tight_layout()
    fname = prod.replace(' ', '_').replace('-', '_') + '_wfcv_forecast.png'
    plt.savefig(fname, dpi=150)
    print(f'{prod}: best = {best_model} (MAPE={best_mape:.1f}%) | saved {fname}')
    plt.close()

In [ ]:
n_horizons = len(horizon_starts)
with pd.ExcelWriter('wfcv_errors.xlsx') as writer:
    for metric in metrics_keys:
        rows = []
        for i in range(n_horizons):
            row = {'Horizon': f'H{i+1}'}
            for prod in products:
                for m in model_names:
                    col = f'{prod}|{m}'
                    row[col] = all_results[prod][m]['per_horizon'][i].get(metric, np.nan)
            rows.append(row)
        df_out = pd.DataFrame(rows).set_index('Horizon')
        df_out.to_excel(writer, sheet_name=metric)
        print(f'{metric} sheet: {df_out.shape[0]} horizons x {df_out.shape[1]} cols')
print('Saved wfcv_errors.xlsx')

In [ ]:
model_names = ['RandomForest', 'XGBoost', 'Lasso', 'LightGBM', 'Prophet']
n_horizons = len(horizon_starts)

with pd.ExcelWriter('wfcv_predictions.xlsx') as writer:
    for prod in products:
        rows = []
        for i in range(n_horizons):
            row = {'Horizon': f'H{i+1}'}
            ref = all_results[prod][model_names[0]]['per_horizon'][i]
            n_steps = len(ref['dates'])
            for s in range(3):
                if s < n_steps:
                    d = ref['dates'][s]
                    row[f'Step{s+1}_Date'] = d.strftime('%Y-%m') if hasattr(d, 'strftime') else str(d)[:7]
                    row[f'Actual_Step{s+1}'] = ref['actuals'][s]
                else:
                    row[f'Step{s+1}_Date'] = ''
                    row[f'Actual_Step{s+1}'] = np.nan
            for m in model_names:
                h = all_results[prod][m]['per_horizon'][i]
                n_preds = len(h['preds'])
                for s in range(3):
                    row[f'{m}_Step{s+1}'] = h['preds'][s] if s < n_preds else np.nan
            rows.append(row)
        df_out = pd.DataFrame(rows).set_index('Horizon')
        sheet_name = prod[:31]
        df_out.to_excel(writer, sheet_name=sheet_name)
        print(f'{sheet_name}: {df_out.shape[0]} horizons x {df_out.shape[1]} cols')
print('Saved wfcv_predictions.xlsx')